# Consistent Character Comics - Indivisual Project (Shehij Raina)

### Imports and Initial Setup

In [ ]:
# Manage Hex Cloud GPU Usage...
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Import Libraries

import accelerate
import google.generativeai as genai
import json
import math
import numpy               as np
import PIL
import random
import textwrap
import torch
import torch.nn.functional as F

from accelerate            import Accelerator
from diffusers             import AutoPipelineForImage2Image, AutoPipelineForText2Image, AutoencoderKL, DDPMScheduler, DPMSolverMultistepScheduler, StableDiffusionPipeline, UNet2DConditionModel
from huggingface_hub       import login
from PIL                   import Image, ImageDraw, ImageFont
from torch.optim           import AdamW
from torch.utils.data      import Dataset, DataLoader
from torchvision           import transforms
from transformers          import CLIPTextModel, CLIPTokenizer
from tqdm.auto             import tqdm

### Helper Functions

In [ ]:
# Helps visualise a set of images
def display_images_in_grid(images, num_row, num_col):
    if len(images) != num_row * num_col:
        raise ValueError("Recheck dimensions for the image grid!")

    width, height = images[0].size # dimensions of an image
    
    grid = Image.new('RGB', size=(num_col * width, num_row * height))
    grid_width, grid_height = grid.size # dimensions of complete grid

    for i, image in enumerate(images):
        grid.paste(image, box=(i % num_col * width, i // num_col * height))

    display(grid)


### Comic Prompt Processing

In [ ]:
COMIC_PROMPT = "Sarah visits the library"
NUM_PANELS   = 4

In [ ]:
# This section processes a textual prompt to generate a structured comic story...

# Here we will extract the description of our character 
# as well as the story to be told through the comic
# using a pretrained LLM such as Google Gemini


# Configure and create the gemini model
genai.configure(api_key="YOUR_API_KEY") # Replace with an actual key
model = genai.GenerativeModel('gemini-1.5-pro')


# Extract character description to isolate their visual identity...
EXTRACT_CHARACTER_QUERY = f'Extract the character description clause from "{COMIC_PROMPT}". No added words or punctuation. If there is no description generate a simple physical description of the character for image generation.'
CHARACTER_PROMPT        = model.generate_content(EXTRACT_CHARACTER_QUERY).text.strip().lower().replace("no character description found.\n\n", "").strip()
print(f'CHARACTER_PROMPT: {CHARACTER_PROMPT}')


# Extract narrative story and remove character description...
EXTRACT_STORY_QUERY     = f'Remove the character description "{CHARACTER_PROMPT}" from the prompt "{COMIC_PROMPT}" (if there is one) and return a simple story prompt.'
STORY_PROPMT = model.generate_content(EXTRACT_STORY_QUERY).text.strip()
print(f'STORY_PROPMT: {STORY_PROPMT}')

# Generate comic-strip captions...
GENERATE_CAPTIONS_QUERY = f'Generate a {NUM_PANELS} panel comic story for the prompt "{STORY_PROPMT}". Provide short simple image generation prompts along with captions for each panel. Do not describe the character in the image generation prompts and refer to them as "<my-character>". Please return your response as valid json, where the panel number is a key and the value is another dictionary with two keys: "prompt" and "caption"."'
response = model.generate_content(GENERATE_CAPTIONS_QUERY).text.strip()

# Process returned string into a python dictionary
json_text            = response.replace("```json", "").replace("```", "").replace("\n", "")
PROMPTS_AND_CAPTIONS = json.loads(json_text)

print(f'PROMPTS_AND_CAPTIONS: {PROMPTS_AND_CAPTIONS}')



In [ ]:
# For the example within the dissertation the generated results were...

# CHARACTER_PROMPT     = "a tall woman with curly red hair wearing a yellow T-shirt and jeans"
# STORY_PROPMT         = "Sarah visits the library"
# PROMPTS_AND_CAPTIONS = {
#   "1": {
#     "prompt": "<my-character> walking into a cozy library with bookshelves and sunlight",
#     "caption": "Sarah steps into the library, ready for a quiet afternoon."
#   },
#   "2": {
#     "prompt": "<my-character> reaching for a book on a shelf in the library",
#     "caption": "She scans the shelves and finds a book that catches her eye."
#   },
#   "3": {
#     "prompt": "<my-character> reading a book at a wooden table in the library",
#     "caption": "Engrossed in the story, she loses track of time."
#   },
#   "4": {
#     "prompt": "<my-character> walking out of the library holding a book, sunset sky",
#     "caption": "With a new adventure in her hands, she heads home happy."
#   }
# }



### Textual Inversion Input Generation 

In [ ]:
# Empty CUDA cache
torch.cuda.empty_cache()

# Log in to HuggingFace if needed...
login(token="YOUR_ACCESS_TOKEN") # Replace with an actual key

# Using the 'lykon/dreamshaper-xl-turbo' model to produce more photorealistic, anatomically accurate, 
# and aesthetically pleasing human characters with consistent facial features and better detail, especially in full-body renderings.
pipeline = AutoPipelineForText2Image.from_pretrained('lykon/dreamshaper-xl-turbo', 
                                                     torch_dtype=torch.float16, 
                                                     variant="fp16")
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline = pipeline.to("cuda")

images_path = "character_images"
os.makedirs(images_path, exist_ok=True)

# Prompt engineering
full_body_prompt   = f'{CHARACTER_PROMPT} , standing facing forward in plain space, full body and head, wide angle view of full body from head to feet, sharp focus, photorealistic'
full_body_negative = "cropped, half body, blurry, bad anatomy, misshapen face, blurry face"
face_prompt        = f'{CHARACTER_PROMPT} , closeup of face, portrait'
face_negative      = "blurry, bad anatomy, misshapen face, blurry face"


num_samples = 5
images = []
for i in range(num_samples):
    # Generate full body images
    image_1 = pipeline(full_body_prompt,
                  num_inference_steps=50, 
                  guidance_scale=8, 
                  negative_prompt=full_body_negative
                 ).images[0]
    # Generate facial images
    image_2 = pipeline(face_prompt,
                      num_inference_steps=50, 
                      guidance_scale=8, 
                      negative_prompt=face_negative
                     ).images[0]
    images.append(image_1)
    images.append(image_2)



grid = display_images_in_grid(images, 2, num_samples)

for i,img in enumerate(images):
    img.save(f'{images_path}/image_{i}.jpg')

del pipeline

# The input images will be saved in the 'character_images' folder

### Using Textual Inversion to Learn the New Character

In [ ]:
BASE_MODEL = "stabilityai/stable-diffusion-2"

# This is the token we will use to represent our new character in image generation prompts (I have used special characters "<",">" to differentiate this token from other real words)
CH_TOKEN = "<my-character>"

In [ ]:
# Load Training Images...

images = []
for filename in os.listdir(images_path):
    if filename.lower().endswith(("jpg", "png", "jpeg")): # Check for file image format
      image_path = os.path.join(images_path, filename)
      images.append(Image.open(image_path).resize((512, 512)))

display_images_in_grid(images, 1, len(images))

In [ ]:
# Textual Inversion Prompt Templates
templates = [
    "a DSLR photo of {}",
    "a casual photo of {}",
    "a candid photo of {}",
    "a close-up photo of {}",
    "a cropped photo of {}",
    "a fashion photo of {}",
    "a good photo of {}",
    "a high-resolution photo of {}",
    "a photo of {}",
    "a photo of {} in a studio",
    "a photo of {} in natural light",
    "a photo of {} looking at the camera",
    "a photo of {} with cinematic lighting",
    "a portrait of {}",
    "a professional photo of {}",
    "a realistic photo of {}",
    "a rendering of {}",
    "a rendition of {}",
    "a studio-lit photo of {}",
    "a picture of {}",
    "an image of {}",
    "an old photo of {}",
    "the photo of {}",
]


In [ ]:
# Setting Up the Dataset...

class CharacterTIDataset(Dataset):
    def __init__(
        self,
        dataset_path,
        tokenizer,
        size=512,
        repeats=100, # Use each image 100 times
        character_token="<my-character>",
    ):

        self.dataset_path    = dataset_path
        self.image_paths     = []
        for filename in os.listdir(self.dataset_path):
            if filename.lower().endswith(("jpg", "png", "jpeg")): # Check for file image format
                self.image_paths.append(os.path.join(self.dataset_path, filename))

        self.num_images      = len(self.image_paths)
        self.templates       = templates

        self.tokenizer       = tokenizer
        self.size            = size
        self.repeats         = repeats
        self.character_token = CH_TOKEN
        
    def __len__(self):
        return self.num_images * self.repeats

    def __getitem__(self, i):
        image = Image.open(self.image_paths[i % self.num_images]).convert("RGB")

        # Resize image
        image = image.resize((self.size, self.size), resample=PIL.Image.BILINEAR)
        # Flip image horizontally (with 0.5 probability)
        image = transforms.RandomHorizontalFlip(p=0.5)(image)
        # Normalise the image's pixel values from the range [0, 255] to [-1, 1]
        image = np.array(image).astype(np.uint8)
        image = ((image / (255/2)) - 1.0).astype(np.float32)
        # Convert the image into a PyTorch tensor
        image = torch.from_numpy(image).permute(2, 0, 1)

        
        # Pick a random text prompt from the templates
        prompt_text = random.choice(self.templates).format(self.character_token)

        dataset_item = {}
        dataset_item["input_ids"] = self.tokenizer(
            prompt_text,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids[0]
        dataset_item["pixels"] = image
        
        return dataset_item

In [ ]:
# Loading the Tokenizer...
tokenizer = CLIPTokenizer.from_pretrained(
    BASE_MODEL,
    subfolder="tokenizer",
)


# Add the new character token (to be trained) in tokenizer...
new_token_added = tokenizer.add_tokens(CH_TOKEN)

if not new_token_added:
    raise ValueError(f'Use a different character token! This tokenizer already contains {CH_TOKEN}.')

In [ ]:
# Loading the Stable Diffusion Model...

text_encoder = CLIPTextModel.from_pretrained(
    BASE_MODEL, 
    subfolder="text_encoder"
)
# Resize token embeddings after adding the new character token in the tokenizer.
# This will create a new embedding vector for the new character token.
text_encoder.resize_token_embeddings(len(tokenizer))


vae = AutoencoderKL.from_pretrained(
    BASE_MODEL, 
    subfolder="vae"
)
unet = UNet2DConditionModel.from_pretrained(
    BASE_MODEL, 
    subfolder="unet"
)

In [ ]:
# We can initialize the embedding for our new token with the one for "person" for a start...
person_token_id = tokenizer.encode("person", add_special_tokens=False)[0]

# Initialise the newly added character token with the embeddings of the word "person"
character_token_id               = tokenizer.convert_tokens_to_ids(CH_TOKEN)
token_embeds                     = text_encoder.get_input_embeddings().weight.data
token_embeds[character_token_id] = token_embeds[person_token_id]

In [ ]:
# Textual inversion involves only training the new embedding vector.
# All other model parameters should be frozen to avoid catastrophic forgetting.
# These other parameters will remain untouched by our training loop.

def freeze_parameters(parameters):
    for p in parameters:
        p.requires_grad = False

# Freeze all the VAE and UNet parameters
freeze_parameters(vae.parameters())
freeze_parameters(unet.parameters())

# Freeze all the Text Encoder parameters except token embeddings
freeze_parameters(text_encoder.text_model.encoder.parameters())
freeze_parameters(text_encoder.text_model.final_layer_norm.parameters())
freeze_parameters(text_encoder.text_model.embeddings.position_embedding.parameters())


In [ ]:
# Create Noise Scheduler for Training...
noise_scheduler = DDPMScheduler.from_pretrained(
    BASE_MODEL,
    subfolder="scheduler"
)

#### Training

In [ ]:
# Setting Up Hyperparameters...

LEARNING_RATE      = 5e-05
NUM_TRAINING_STEPS = 4000
BATCH_SIZE         = 4
GRAD_ACC_STEPS     = 1
OUTPUT_DIR         = "character-ti"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Whether or not to use the character description prompt while training
TRAIN_WITH_TEXT_PROMPT = True

In [ ]:
# Training function

def training_function(text_encoder, vae, unet):
    batch_size                  = BATCH_SIZE
    gradient_accumulation_steps = GRAD_ACC_STEPS
    learning_rate               = LEARNING_RATE
    max_train_steps             = NUM_TRAINING_STEPS
    output_dir                  = OUTPUT_DIR

    accelerator = Accelerator(
        gradient_accumulation_steps=gradient_accumulation_steps,
        mixed_precision="fp16"
    )

    text_encoder.gradient_checkpointing_enable()
    unet.enable_gradient_checkpointing()

    # Prepare dataset and dataloader...
    dataset = CharacterTIDataset(
          dataset_path=images_path,
          tokenizer=tokenizer,
          size=vae.config.sample_size,
          character_token=CH_TOKEN,
          repeats=100,
    )
    train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Scale the learning rate
    learning_rate = (learning_rate * gradient_accumulation_steps * batch_size * accelerator.num_processes)

    # Initialize the optimizer
    optimizer = AdamW(
        text_encoder.get_input_embeddings().parameters(),  # only optimize the embeddings
        lr=learning_rate,
    )

    text_encoder, optimizer, train_dataloader = accelerator.prepare(
        text_encoder, optimizer, train_dataloader
    )

    # Move VAE and UNet to device
    vae.to(accelerator.device, dtype=torch.float16)
    unet.to(accelerator.device, dtype=torch.float16)
    vae.eval()
    unet.train()

    
    num_train_epochs = math.ceil(max_train_steps / (math.ceil(len(train_dataloader) / gradient_accumulation_steps)))

    # Progress bar
    progress_bar = tqdm(range(max_train_steps), disable=not accelerator.is_local_main_process)
    progress_bar.set_description("Steps")
    global_step = 0

    for epoch in range(num_train_epochs):
        text_encoder.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(text_encoder):
                # Convert images to latent space
                latents = vae.encode(batch["pixels"].to(dtype=torch.float16)).latent_dist.sample().detach()
                latents = latents * 0.18215

                # Add noise to latents
                noise         = torch.randn_like(latents)
                timesteps     = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device).long() # random timestep
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # Get the text embedding for conditioning
                text_embedding = text_encoder(batch["input_ids"])[0]

                # Predict the noise
                noise_pred = unet(noisy_latents, timesteps, text_embedding.to(torch.float16)).sample

                 # Get the target for loss depending on the prediction type
                if noise_scheduler.config.prediction_type == "epsilon":
                    target = noise
                elif noise_scheduler.config.prediction_type == "v_prediction":
                    target = noise_scheduler.get_velocity(latents, noise, timesteps)
                else:
                    raise ValueError(f"Unknown prediction type")

                loss = F.mse_loss(noise_pred, target, reduction="none").mean([1, 2, 3]).mean()

                if TRAIN_WITH_TEXT_PROMPT: 
                    with torch.no_grad():
                        # Get embedding for image caption e.x. "A photo of {CH_TOKEN}"
                        char_input_ids = batch["input_ids"]
                        char_embedding = text_encoder(char_input_ids)[0]
                        char_embedding_mean = char_embedding.mean(dim=1)  # shape: (batch_size, hidden_dim)
                
                        # Get embedding for the fixed character description
                        target_ids       = tokenizer(CHARACTER_PROMPT, return_tensors="pt").input_ids.to(accelerator.device)
                        target_embedding = text_encoder(target_ids)[0]
                        target_embedding_mean = target_embedding.mean(dim=1)  # shape: (1, hidden_dim)
                
                    # Compute cosine similarity between each char_embedding and the fixed description
                    embedding_loss = 1 - F.cosine_similarity(
                        char_embedding_mean, 
                        target_embedding_mean.expand_as(char_embedding_mean)
                    ).mean()

                    loss = loss + (0.1 * embedding_loss)


                accelerator.backward(loss)

                # Zero out the gradients for all token embeddings except the newly added character embeddings
                if accelerator.num_processes > 1:
                    grads = text_encoder.module.get_input_embeddings().weight.grad
                else:
                    grads = text_encoder.get_input_embeddings().weight.grad
                    
                # Get the index for tokens that we want to zero the grads for
                index_grads_to_zero                = (torch.arange(len(tokenizer)) != character_token_id)
                grads.data[index_grads_to_zero, :] = grads.data[index_grads_to_zero, :].fill_(0)

                
                optimizer.step()
                optimizer.zero_grad()

            if accelerator.sync_gradients:
                # Update progress bar...
                progress_bar.update(1)
                global_step += 1

                # Save embeddings every 500 steps
                if global_step % 500 == 0: 
                    save_path           = os.path.join(output_dir, f"learned_embeds-step-{global_step}.bin")
                    learned_embeds      = accelerator.unwrap_model(text_encoder).get_input_embeddings().weight[character_token_id]
                    learned_embeds_dict = {CH_TOKEN: learned_embeds.detach().cpu()}
                    torch.save(learned_embeds_dict, save_path)


            logs = {"loss": loss.detach().item()}
            progress_bar.set_postfix(**logs)

            # Training completed!
            if global_step >= max_train_steps:
                break

        accelerator.wait_for_everyone()


    # Save the whole pipeline...
    if accelerator.is_main_process:
        pipeline = StableDiffusionPipeline.from_pretrained(
            BASE_MODEL,
            text_encoder=accelerator.unwrap_model(text_encoder),
            tokenizer=tokenizer,
            vae=vae,
            unet=unet,
        )
        pipeline.save_pretrained(output_dir)
        # Save trained embeddings seperately
        save_path = os.path.join(output_dir, f"learned_embeds.bin")
        learned_embeds      = accelerator.unwrap_model(text_encoder).get_input_embeddings().weight[character_token_id]
        learned_embeds_dict = {CH_TOKEN: learned_embeds.detach().cpu()}
        torch.save(learned_embeds_dict, save_path)



accelerate.notebook_launcher(training_function, args=(text_encoder, vae, unet), num_processes=1)

for parameter in unet.parameters():
    if parameter.grad is not None:
        del parameter.grad

for parameter in text_encoder.parameters():
    if parameter.grad is not None:
        del param.grad

torch.cuda.empty_cache()


### Generating Comic-Strip

In [ ]:
torch.cuda.empty_cache()

# Set up the Stable Diffusion Pipeline
pipeline = StableDiffusionPipeline.from_pretrained(
    OUTPUT_DIR,
    scheduler=DPMSolverMultistepScheduler.from_pretrained(OUTPUT_DIR, subfolder="scheduler"),
    torch_dtype=torch.float16,
).to("cuda")

In [ ]:
torch.cuda.empty_cache()

os.makedirs('generated_images', exist_ok=True)
os.makedirs('comic_images',     exist_ok=True)

negative_prompt = "blurry, low quality, low resolution, deformed, disfigured, mutated, extra limbs, missing limbs, broken anatomy, bad anatomy, bad proportions, cloned face, duplicate face, ugly, artifact, grainy, jpeg artifacts, watermark, text, logo, distorted, long neck, fused fingers, extra fingers, missing fingers, poorly drawn face, poorly drawn hands, bad hands, bad face"

#Generate Images...
generated_images = []
for prompt, caption in PROMPTS_AND_CAPTIONS.items():
    image = pipeline(prompt,
                  num_inference_steps=40,
                  guidance_scale=8,
                  negative_prompt=negative_prompt
                 ).images[0]
    image.save(f'generated_images/{prompt}.png')
    generated_images.append(image)
display_images_in_grid(generated_images, 1, NUM_PANELS)





In [ ]:
# Image-to-Image Style Transfer
# Convert the photorealistic panel images into an image that looks hand-drawn...

def convert_to_art(image):
    # Load Stable Diffusion pipeline...
    pipeline                    = AutoPipelineForImage2Image.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
    pipeline.scheduler          = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
    pipeline.safety_checker = None
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    pipeline.to(device)
    
    
    # Load IP-Adapter
    pipeline.load_ip_adapter(
        "h94/IP-Adapter",
        subfolder="models",
        weight_name="ip-adapter_sd15.bin"
    )
    pipeline.set_ip_adapter_scale(0.5)
    
    # Load LoRA
    pipeline.unet = PeftModel.from_pretrained(pipeline.unet, "ghibli_lora")
    pipeline.unet.eval()
    
    # Process Image
    image = image.convert("RGB")
    
    # Size 512X512 for SD...
    image_sd = transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR)(image)
    image_sd = transforms.CenterCrop(512)(image)
    # Size 224x224 for the IP-Adapter...
    image_ip = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)(image)
    image_ip = transforms.CenterCrop(224)(image_ip)
    image_ip = transforms.ToTensor()(image_ip).unsqueeze(0).to("cuda")
    
    
    # Run the style transfer...
    with torch.autocast("cuda"):
        image = pipeline(
            prompt="A Ghibli style scene", # Prompt contains "Ghibli style" so that generated image is of that style
            image=image_sd,
            ip_adapter_image=image_ip,
            num_inference_steps=40,
            strength=0.8, # decrease value if too much of the original image is lost
            guidance_scale=8.5,
            negative_prompt="blurry face"
        ).images[0]

    del pipeline
    
    return image

In [ ]:
# Convert to hand-drawn art...
stylized_images = []
for image in generated_images:
    stylized_image = convert_to_art(image)
    stylized_image.save(f'comic_images/{prompt}.png')
    stylized_images.append(stylized_image)
display_images_in_grid(stylized_images, 1, NUM_PANELS)


In [ ]:
# Assemble Comic Strip!!!

def create_comic_strip(images, captions, font_path="Melon Camp.ttf", panel_width=512, font_size=30, padding=20, gutter=20):
    font = ImageFont.truetype(font_path, font_size)

    num_panels = len(images) # NUM_PANELS
    cols       = math.ceil(math.sqrt(num_panels))
    rows       = math.ceil(num_panels / cols)

    # Estimate average image height after resizing
    aspect_ratios   = [img.height / img.width for img in images]
    resized_heights = [int(panel_width * ar) for ar in aspect_ratios]

    # Text wrapping configuration
    line_height    = font.getbbox("A")[3] + 5
    char_width     = font.getbbox("A")[2] - font.getbbox("A")[0]
    chars_per_line = (panel_width - 2 * padding) // char_width

    wrapped_captions   = []
    max_caption_height = 0
    for caption in captions:
        wrapped        = textwrap.fill(caption, width=chars_per_line)
        lines          = wrapped.count('\n') + 1
        wrapped_captions.append(wrapped)
        max_caption_height = max(max_caption_height, lines * line_height)

    # Max panel height (based on tallest resized image)
    max_image_height = max(resized_heights)
    panel_height = max_image_height + max_caption_height + padding

    # Total comic strip dimensions with padding
    total_width = cols * panel_width + (cols - 1) * gutter + 2 * padding
    total_height = rows * panel_height + (rows - 1) * gutter + 2 * padding

    comic_strip = Image.new("RGB", (total_width, total_height), color="white")
    draw = ImageDraw.Draw(comic_strip)

    for idx, (img, caption) in enumerate(zip(images, wrapped_captions)):
        row = idx // cols
        col = idx % cols

        x = padding + col * (panel_width + gutter)
        y = padding + row * (panel_height + gutter)

        # Resize image to fit panel_width
        aspect_ratio = img.height / img.width
        new_height   = int(panel_width * aspect_ratio)
        resized_img  = img.resize((panel_width, new_height), Image.LANCZOS)

        comic_strip.paste(resized_img, (x, y))

        # Draw caption with centered alignment
        for j, line in enumerate(caption.split("\n")):
            line_width = font.getbbox(line)[2] - font.getbbox(line)[0]
            text_x     = x + (panel_width - line_width) // 2
            text_y     = y + resized_img.height + 5 + j * line_height
            draw.text((text_x, text_y), line, fill="black", font=font)

    return comic_strip

captions = [i["caption"] for i in PROMPTS_AND_CAPTIONS.values()]
comic = create_comic_strip(stylized_images, captions)
display(comic)
comic.save("comic.png")
